# 数独压力测试 · 谁能保住正确答案，又能修正错误？

本实验比较普通循环、全通道门控和分区门控。沿用第一轮正式实验的 2,048 道训练题和训练代码。相同种子的三组模型使用相同的主干初值、采样顺序与数字置换。每组各用种子 17、29、43 从头完成 5,000 次 AdamW 更新。训练时展开 8 轮，评估时继续到第 32 轮。

**两个独立的新题组**　原分布与难题各 512 道。难题由基础求解器在只用裸单和隐单、反复运用到无新解时仍留有空格来定义。每题只有一个解。题面、答案与顺序在训练前固定，均收录在下方代码和本次结果中。

**主要对照**　只看难题，对同一训练种子的分区门控与全通道门控逐格配对。对于第 8 轮双方都正确的同一批空格，统计第 9–32 轮中曾经改错的比例。对于第 8 轮双方都错误的同一批空格，统计第 32 轮已经改对的比例。再与普通循环对照，并在原分布上重复计算；同时记录中途修正和整盘表现。

报告同时给出三个种子的配对差和按题重采样的区间。区间针对这三组已训练权重。采用零差值为参照，未预设可接受的纠错率下降幅度。若纠错差值的区间跨零，仍有可能存在纠错代价。

**运行**　选择 Colab GPU，点击“全部运行”，按提示输入当前资源面板的 CU/小时。各次运行单独保留可校验、可恢复的轨迹和模型权重。最后一个单元汇总并释放 GPU。


In [1]:
import time
if 'STRESS_SESSION_START' not in globals():
    STRESS_SESSION_START=time.monotonic()


In [2]:
# 可离线生成的 9×9 数独。导出为 notebook 单元后只依赖 Python 和 NumPy。
import hashlib
import random
import numpy as np

_SDK_ALL = (1 << 9) - 1
_SDK_ROW = tuple(i // 9 for i in range(81))
_SDK_COL = tuple(i % 9 for i in range(81))
_SDK_BOX = tuple(i // 27 * 3 + i % 9 // 3 for i in range(81))


def _sdk_completion(puzzle, *, known=None, rng=None):
    """Return one completion, excluding known when supplied. Internal grids are valid."""
    board = list(puzzle)
    rows, cols, boxes = [0] * 9, [0] * 9, [0] * 9
    empty = []
    for i, digit in enumerate(board):
        if digit:
            bit = 1 << (digit - 1)
            rows[_SDK_ROW[i]] |= bit
            cols[_SDK_COL[i]] |= bit
            boxes[_SDK_BOX[i]] |= bit
        else:
            empty.append(i)
    if rng is not None:
        rng.shuffle(empty)

    def search(depth, differs):
        if depth == len(empty):
            return board.copy() if known is None or differs else None
        best, best_mask, best_n = depth, 0, 10
        for j in range(depth, len(empty)):
            i = empty[j]
            mask = _SDK_ALL & ~(rows[_SDK_ROW[i]] | cols[_SDK_COL[i]] | boxes[_SDK_BOX[i]])
            count = mask.bit_count()
            if count < best_n:
                if count == 0:
                    return None
                best, best_mask, best_n = j, mask, count
                if count == 1:
                    break
        empty[depth], empty[best] = empty[best], empty[depth]
        i = empty[depth]
        r, c, b = _SDK_ROW[i], _SDK_COL[i], _SDK_BOX[i]
        candidates = []
        while best_mask:
            bit = best_mask & -best_mask
            candidates.append(bit)
            best_mask ^= bit
        if best_n > 1:
            if rng is not None:
                rng.shuffle(candidates)
            elif known is not None and not differs:
                preferred_last = 1 << (known[i] - 1)
                if preferred_last in candidates:
                    candidates.remove(preferred_last)
                    candidates.append(preferred_last)
        for bit in candidates:
            digit = bit.bit_length()
            board[i] = digit
            rows[r] |= bit
            cols[c] |= bit
            boxes[b] |= bit
            result = search(depth + 1, differs or (known is not None and digit != known[i]))
            if result is not None:
                return result
            rows[r] ^= bit
            cols[c] ^= bit
            boxes[b] ^= bit
        board[i] = 0
        empty[depth], empty[best] = empty[best], empty[depth]
        return None

    return search(0, False)


def _sdk_split_id(solution):
    # The cycle lengths of the digit permutation formed by two rows survive
    # digit relabeling and column permutation. Sorting both orientations also
    # makes the signature invariant to row order and transposition. Thus every
    # board in a Sudoku symmetry orbit is assigned to the same split. Distinct
    # orbits can share a signature, which only keeps extra examples together.
    rows = [solution[i : i + 9] for i in range(0, 81, 9)]

    def fingerprint(lines):
        result = []
        for i, left in enumerate(lines):
            for right in lines[i + 1 :]:
                perm = [0] * 10
                for a, b in zip(left, right):
                    perm[a] = b
                seen, lengths = 0, []
                for digit in range(1, 10):
                    length = 0
                    while not (seen & (1 << digit)):
                        seen |= 1 << digit
                        digit = perm[digit]
                        length += 1
                    if length:
                        lengths.append(length)
                result.append(tuple(sorted(lengths)))
        return tuple(sorted(result))

    signature = sorted((fingerprint(rows), fingerprint(list(zip(*rows)))))
    digest = hashlib.blake2b(repr(signature).encode("ascii"), digest_size=8).digest()
    bucket = int.from_bytes(digest, "little") % 10
    return "val" if bucket == 8 else "test" if bucket == 9 else "train"


def _sdk_puzzle(solution, rng, index):
    # Mix starting clue densities. Clues added to force uniqueness count too.
    low, high = ((24, 31), (32, 40), (43, 52))[index % 3]
    puzzle = [0] * 81
    for i in rng.sample(range(81), rng.randint(low, high)):
        puzzle[i] = solution[i]
    while True:
        alternative = _sdk_completion(puzzle, known=solution)
        if alternative is None:
            return puzzle
        i = rng.choice([i for i in range(81) if alternative[i] != solution[i]])
        puzzle[i] = solution[i]


def make_sudoku_splits(n_train=2048, n_val=256, n_test=256, seed=2026):
    """Return {train, val, test: (puzzles[N,81], solutions[N,81])}, uint8."""
    sizes = dict(train=n_train, val=n_val, test=n_test)
    if any(not isinstance(n, (int, np.integer)) or n < 0 for n in sizes.values()):
        raise ValueError("Split sizes must be nonnegative integers.")
    rng = random.Random(int(seed))
    splits = {k: (np.empty((n, 81), np.uint8), np.empty((n, 81), np.uint8))
              for k, n in sizes.items()}
    counts = dict.fromkeys(sizes, 0)
    seen = set()
    while counts != sizes:
        solution = _sdk_completion([0] * 81, rng=rng)
        puzzle_seed = rng.getrandbits(64)
        key = _sdk_split_id(solution)
        i = counts[key]
        code = bytes(solution)
        if i == sizes[key] or code in seen:
            continue
        seen.add(code)
        puzzle = _sdk_puzzle(solution, random.Random(puzzle_seed), i)
        splits[key][0][i] = puzzle
        splits[key][1][i] = solution
        counts[key] += 1
    return splits


def sample_sudoku_batch(split, batch_size, rng, augment=True):
    """Sample with replacement. Pass a seeded np.random.Generator. No split mutation."""
    puzzles, solutions = split
    picks = rng.integers(0, len(puzzles), size=batch_size)
    puzzles, solutions = puzzles[picks], solutions[picks]
    if not augment:
        return puzzles, solutions

    def shuffled_axis():
        return np.concatenate([3 * band + rng.permutation(3) for band in rng.permutation(3)])

    grid = np.arange(81).reshape(9, 9)
    if rng.integers(2):
        grid = grid.T
    positions = grid[np.ix_(shuffled_axis(), shuffled_axis())].reshape(-1)
    digit_map = np.zeros((batch_size, 10), np.uint8)
    digit_map[:, 1:] = 1 + np.argsort(rng.random((batch_size, 9)), axis=1)
    return tuple(np.take_along_axis(digit_map, values[:, positions], axis=1)
                 for values in (puzzles, solutions))


In [3]:
import torch
from torch import nn
from torch.nn import functional as F


class SudokuBlock(nn.Module):
    """One shared pre-norm Transformer block with bidirectional attention."""

    def __init__(self, width, heads, ffn_width):
        super().__init__()
        self.heads = heads
        self.head_width = width // heads
        self.attn_norm = nn.LayerNorm(width)
        self.qkv = nn.Linear(width, 3 * width)
        self.attn_out = nn.Linear(width, width)
        positions = torch.arange(81)
        rows, cols = positions // 9, positions % 9
        boxes = (rows // 3) * 3 + cols // 3
        same_row = rows[:, None] == rows[None, :]
        same_col = cols[:, None] == cols[None, :]
        same_box = boxes[:, None] == boxes[None, :]
        global_attention = torch.ones_like(same_row)
        head_types = (same_row, same_col, same_box, global_attention)
        masks = [head_types[i % 4] for i in range(heads)]
        if heads < 4:
            masks[-1] = global_attention
        # SDPA True entries permit attention. Shared by all variants, this
        # puzzle-topology prior contains only locations, never puzzle answers.
        self.register_buffer("attention_mask", torch.stack(masks)[None], persistent=False)
        self.ffn_norm = nn.LayerNorm(width)
        self.ffn = nn.Sequential(
            nn.Linear(width, ffn_width), nn.GELU(), nn.Linear(ffn_width, width)
        )

    def forward(self, state):
        batch, cells, width = state.shape
        qkv = self.qkv(self.attn_norm(state))
        qkv = qkv.reshape(batch, cells, 3, self.heads, self.head_width)
        query, key, value = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        context = F.scaled_dot_product_attention(
            query, key, value, attn_mask=self.attention_mask, is_causal=False
        )
        context = context.transpose(1, 2).reshape(batch, cells, width)
        proposal = state + self.attn_out(context)
        return proposal + self.ffn(self.ffn_norm(proposal))


class SudokuLoopTransformer(nn.Module):
    """Inputs are digits 0–9 (0 = blank). Logit indices 0–8 denote digits 1–9."""

    def __init__(self, variant, width=96, heads=4, ffn_width=192, core_width=32):
        super().__init__()
        if variant not in ("plain", "full_gate", "split_gate"):
            raise ValueError(f"Unknown model variant: {variant}")
        if heads < 1 or width % heads or not 0 < core_width < width:
            raise ValueError("Use a width divisible by heads and 0 < core_width < width.")
        self.variant = variant
        self.digit = nn.Embedding(10, width)
        self.row = nn.Embedding(9, width)
        self.col = nn.Embedding(9, width)
        self.box = nn.Embedding(9, width)
        cell = torch.arange(81)
        rows, cols = cell // 9, cell % 9
        self.register_buffer("rows", rows, persistent=False)
        self.register_buffer("cols", cols, persistent=False)
        self.register_buffer("boxes", (rows // 3) * 3 + cols // 3, persistent=False)
        self.block = SudokuBlock(width, heads, ffn_width)
        self.readout = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 9))
        if variant != "plain":
            self.gate = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, core_width))
            # A full-stream gate shares one decision per group of channels. Both
            # controls have the same number of learned gates and the same GEMMs.
            channel = torch.arange(width)
            self.register_buffer("gate_group", channel % core_width, persistent=False)
            gated = channel < core_width if variant == "split_gate" else channel >= 0
            self.register_buffer("gated_channels", gated, persistent=False)

    def initial_state(self, puzzles):
        xy = self.row(self.rows) + self.col(self.cols) + self.box(self.boxes)
        return 0.5 * (self.digit(puzzles.long()) + xy)

    def step(self, state, gate_mode="learned"):
        proposal = self.block(state)
        if self.variant == "plain":
            return proposal
        gate = torch.sigmoid(self.gate(proposal)).to(dtype=state.dtype)
        if gate_mode == "open":
            gate = torch.ones_like(gate)
        elif gate_mode == "closed":
            gate = torch.zeros_like(gate)
        elif gate_mode != "learned":
            raise ValueError(f"Unknown gate intervention: {gate_mode}")
        gate = gate.index_select(-1, self.gate_group)
        # Only this interpolation can write the protected channels. Attention
        # and FFN proposals can read all channels. The readout never writes back.
        gated_proposal = torch.lerp(state, proposal, gate)
        return torch.where(self.gated_channels, gated_proposal, proposal)

    def forward(self, puzzles, steps, return_all=False, gate_mode="learned"):
        if puzzles.ndim != 2 or puzzles.shape[1] != 81 or steps < 1:
            raise ValueError("Expected [batch, 81] puzzles and steps >= 1.")
        state = self.initial_state(puzzles)
        if return_all:
            logits = []
            for _ in range(steps):
                state = self.step(state, gate_mode=gate_mode)
                logits.append(self.readout(state))
            return torch.stack(logits, dim=1)  # [batch, steps, 81, 9]
        for _ in range(steps):
            state = self.step(state, gate_mode=gate_mode)
        return self.readout(state)  # [batch, 81, 9]


In [4]:
import math
import time
from contextlib import nullcontext

import torch
from torch.nn import functional as F


FORMAL_VARIANTS = ("plain", "full_gate", "split_gate")
FORMAL_SEEDS = (17, 29, 43)
FORMAL_DEPTHS = (8, 16, 32)


@torch.inference_mode()
def evaluate_boards(model, split, device, precision, batch_size=64, check_budget=None):
    """Each column is in source-puzzle order. Givens are copied when scoring a board."""
    model.eval()
    x_all, y_all = split
    if len(x_all) == 0 or len(x_all) != len(y_all):
        raise ValueError("Evaluation needs a nonempty, aligned split.")
    # Batch all CPU transfers per minibatch; keep counts as native Python ints.
    batches = []
    for start in range(0, len(x_all), batch_size):
        if check_budget:
            check_budget()
        x = torch.as_tensor(x_all[start:start + batch_size], device=device).long()
        y = torch.as_tensor(y_all[start:start + batch_size], device=device).long()
        blank = x.eq(0)
        cols = [blank.sum(1)]
        finite = []
        with precision():
            state = model.initial_state(x)
            for d in range(1, FORMAL_DEPTHS[-1] + 1):
                state = model.step(state)
                if d not in FORMAL_DEPTHS:
                    continue
                logits = model.readout(state)
                finite.append(torch.isfinite(logits).all())
                correct = logits.argmax(-1).add(1).eq(y) & blank
                if d == FORMAL_DEPTHS[0]:
                    right8 = correct
                right = correct.sum(1)
                ruined = (right8 & ~correct).sum(1)
                repaired = (correct & ~right8).sum(1)
                cols.extend((right, ruined, repaired))
        if not torch.stack(finite).all().item():
            raise FloatingPointError("Evaluation produced nonfinite logits.")
        batches.append(torch.stack(cols, dim=1).cpu())
    values = torch.cat(batches)
    n_blank = values[:, 0]
    at_eight = values[:, 1]
    result = {"n_blank": n_blank.tolist(), "right8": at_eight.tolist(), "by_depth": {}}
    for j, depth in enumerate(FORMAL_DEPTHS):
        offset = j * 3 + 1
        right, ruined, repaired = values[:, offset:offset + 3].unbind(1)
        if not torch.equal(right, at_eight - ruined + repaired):
            raise AssertionError("Per-board flip counts do not reconcile.")
        result["by_depth"][str(depth)] = {
            "exact": right.eq(n_blank).tolist(), "right": right.tolist(),
            "ruined": ruined.tolist(), "repaired": repaired.tolist(),
        }
    return result


def fit_formal_arm(variant, seed, splits, *, device="cuda", check_budget=None,
                   progress=None, total_steps=5000):
    """Train one fixed-recipe arm; the caller chooses its held-out readouts."""
    if variant not in FORMAL_VARIANTS or seed not in FORMAL_SEEDS or total_steps < 1:
        raise ValueError("Invalid formal variant, seed or step count.")
    device = torch.device(device)
    cuda = device.type == "cuda"
    if cuda:
        torch.cuda.set_device(device.index if device.index is not None else torch.cuda.current_device())
    torch.set_float32_matmul_precision("high")
    dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if cuda else torch.float32
    precision = (lambda: torch.autocast("cuda", dtype=dtype)) if cuda else nullcontext
    sync = (lambda: torch.cuda.synchronize(device)) if cuda else (lambda: None)
    torch.manual_seed(seed)
    model = SudokuLoopTransformer(variant, width=96, heads=4, ffn_width=192, core_width=32).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    scaled = cuda and dtype == torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=scaled, init_scale=256.0)
    batch_seed = seed + 1010
    gen = torch.Generator(device=device).manual_seed(batch_seed)
    train_x, train_y = [torch.as_tensor(a, device=device).long() for a in splits["train"]]
    if len(train_x) == 0 or train_x.shape != train_y.shape:
        raise ValueError("Training needs a nonempty, aligned split.")

    model.train()
    sync()
    started = time.perf_counter()
    overflow_retries = 0
    for step in range(1, total_steps + 1):
        if check_budget:
            check_budget()
        ids = torch.randint(len(train_x), (64,), device=device, generator=gen)
        x, y = train_x[ids], train_y[ids]
        digit_map = torch.rand((64, 9), device=device, generator=gen).argsort(1) + 1
        digit_map = torch.cat((torch.zeros_like(digit_map[:, :1]), digit_map), dim=1)
        x, y = digit_map.gather(1, x), digit_map.gather(1, y)
        target = (y - 1).masked_fill(x.ne(0), -100)
        # FP16 overflow retries reuse this very batch and never count as updates.
        for attempt in range(9):
            opt.zero_grad(set_to_none=True)
            with precision():
                logits = model(x, steps=8)
                loss = F.cross_entropy(logits.reshape(-1, 9), target.reshape(-1))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if (torch.isfinite(loss.detach()) & torch.isfinite(grad_norm)).item():
                scaler.step(opt)
                scaler.update()
                break
            if not scaled or attempt == 8:
                raise FloatingPointError(f"Nonfinite training value: {variant}, seed {seed}, update {step}")
            if check_budget:
                check_budget()
            scaler.update(new_scale=scaler.get_scale() / 2.0)
            overflow_retries += 1
        if progress and (step % 1000 == 0 or step == total_steps):
            progress(variant, seed, step, total_steps)
    sync()
    train_seconds = time.perf_counter() - started
    final_loss = float(loss.detach())
    if not math.isfinite(final_loss) or not all(torch.isfinite(p).all().item() for p in model.parameters()):
        raise FloatingPointError("Final trained parameters or loss are nonfinite.")
    if not all(int(opt.state[p]["step"].item()) == total_steps for p in model.parameters()):
        raise RuntimeError("Optimizer step counters do not match the completed updates.")
    record = {
        "variant": variant, "seed": seed, "batch_seed": batch_seed, "steps": total_steps,
        "parameters": sum(p.numel() for p in model.parameters()), "last_loss": final_loss,
        "precision": str(dtype), "torch_version": str(torch.__version__),
        "gpu": torch.cuda.get_device_name(device) if cuda else "CPU",
        "train_seconds": train_seconds, "updates_per_second": total_steps / train_seconds,
        "fp16_overflow_retries": overflow_retries, "splits": {}, "eval_split_seconds": {},
    }
    return model, record, device, precision, sync


def train_one(variant, seed, splits, *, device="cuda", save_run=None, check_budget=None,
              progress=None, total_steps=5000):
    """Train a fresh fixed-recipe arm. A completed record contains only JSON values."""
    model, record, device, precision, sync = fit_formal_arm(
        variant, seed, splits, device=device, check_budget=check_budget,
        progress=progress, total_steps=total_steps
    )
    for split_name in ("val", "test"):
        sync()
        before = time.perf_counter()
        record["splits"][split_name] = evaluate_boards(
            model, splits[split_name], device, precision, batch_size=64, check_budget=check_budget
        )
        sync()
        record["eval_split_seconds"][split_name] = time.perf_counter() - before
    record["eval_seconds"] = sum(record["eval_split_seconds"].values())
    if save_run:
        save_run(record)
    return record


In [5]:
import base64
import hashlib
import io
import os
from pathlib import Path
import tempfile
import time
import zlib

import numpy as np


_STRESS_CODEC = "u8-xor-rounds-blank-major-zlib-base64-v1"


def _trajectory_grids(split):
    puzzles, solutions = map(np.asarray, split)
    if (puzzles.ndim != 2 or puzzles.shape[1] != 81 or not len(puzzles)
            or puzzles.shape != solutions.shape):
        raise ValueError("Expected matching, nonempty [n,81] puzzle and solution arrays.")
    if (puzzles.dtype.kind not in "ui" or solutions.dtype.kind not in "ui"
            or puzzles.min() < 0 or puzzles.max() > 9
            or solutions.min() < 1 or solutions.max() > 9
            or np.any((puzzles != 0) & (puzzles != solutions))):
        raise ValueError("Invalid Sudoku digit values or conflicting givens.")
    return (np.ascontiguousarray(puzzles, dtype=np.uint8),
            np.ascontiguousarray(solutions, dtype=np.uint8))


def _trajectory_grid_hash(puzzles, solutions):
    # Hash version + board count (unsigned little-endian 64-bit) + C-order
    # puzzle bytes + C-order solution bytes. This binds the entire board order.
    digest = hashlib.sha256(b"sudoku-grid-order-v1\0" + len(puzzles).to_bytes(8, "little"))
    digest.update(memoryview(puzzles))
    digest.update(memoryview(solutions))
    return digest.hexdigest()


def pack_blank_predictions(prediction, split):
    """Losslessly encode [round, blank]. Blank order follows puzzles[puzzles==0]."""
    puzzles, solutions = _trajectory_grids(split)
    n_blanks = int(np.count_nonzero(puzzles == 0))
    values = np.asarray(prediction)
    if values.dtype.kind not in "ui" or values.shape != (32, n_blanks):
        raise ValueError("Expected 32 rows and one column per originally blank cell.")
    if n_blanks and (values.min() < 1 or values.max() > 9):
        raise ValueError("Predictions must use digits 1 through 9.")
    values = np.ascontiguousarray(values, dtype=np.uint8)
    # Store the first guess and XORs with the previous guess. Stable parts of a
    # cell's path then become runs of zeros even for sets wider than 32 KiB.
    delta = np.empty_like(values)
    delta[0] = values[0]
    np.bitwise_xor(values[1:], values[:-1], out=delta[1:])
    encoded = base64.b64encode(zlib.compress(delta.T.tobytes(order="C"), 9)).decode("ascii")
    return {
        "schema": _STRESS_CODEC, "shape": [32, n_blanks], "boards": len(puzzles),
        "split_sha256": _trajectory_grid_hash(puzzles, solutions),
        "raw_sha256": hashlib.sha256(memoryview(values)).hexdigest(), "data": encoded,
    }


def decode_blank_predictions(packed, split):
    """Restore all 32 argmax digits, checking source order, byte counts and digest."""
    puzzles, solutions = _trajectory_grids(split)
    n_blanks = int(np.count_nonzero(puzzles == 0))
    if (packed["schema"] != _STRESS_CODEC or packed["boards"] != len(puzzles)
            or packed["shape"] != [32, n_blanks]
            or packed["split_sha256"] != _trajectory_grid_hash(puzzles, solutions)):
        raise ValueError("Trajectory schema, shape or source split hash mismatch.")
    encoded = base64.b64decode(packed["data"], validate=True)
    delta = zlib.decompress(encoded)
    if len(delta) != 32 * n_blanks:
        raise ValueError("Incorrect decompressed trajectory length.")
    by_blank = np.frombuffer(delta, dtype=np.uint8).reshape(n_blanks, 32)
    pred = np.ascontiguousarray(np.bitwise_xor.accumulate(by_blank, axis=1).T)
    if hashlib.sha256(memoryview(pred)).hexdigest() != packed["raw_sha256"]:
        raise ValueError("Decoded trajectory SHA-256 mismatch.")
    if n_blanks and (pred.min() < 1 or pred.max() > 9):
        raise ValueError("Out-of-range digit in decoded trajectory.")
    return pred


def capture_blank_predictions(model, split, device, precision, batch_size=64, check_budget=None):
    import torch

    model.eval()
    puzzles, solutions = _trajectory_grids(split)
    prediction = np.empty((32, np.count_nonzero(puzzles == 0)), dtype=np.uint8)
    offset = 0
    with torch.inference_mode():
        for start in range(0, len(puzzles), batch_size):
            if check_budget:
                check_budget()
            chunk = puzzles[start:start + batch_size]
            x = torch.as_tensor(chunk, device=device).long()
            digits, finite = [], []
            with precision():
                state = model.initial_state(x)
                for _ in range(32):
                    state = model.step(state)
                    logits = model.readout(state)
                    finite.append(torch.isfinite(logits).all())
                    digits.append(logits.argmax(-1).add(1).to(torch.uint8))
            if not torch.stack(finite).all().item():
                raise FloatingPointError("Nonfinite trajectory logits.")
            blank_mask = chunk == 0
            end = offset + np.count_nonzero(blank_mask)
            prediction[:, offset:end] = torch.stack(digits).cpu().numpy()[:, blank_mask]
            offset = end
    if offset != prediction.shape[1]:
        raise RuntimeError("Trajectory blank count mismatch.")
    return pack_blank_predictions(prediction, (puzzles, solutions))


def _save_stress_weights(model, directory, variant, seed, steps):
    import torch

    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    final = directory / f"sudoku-{variant}-seed{seed}-steps{steps}.pt"
    weights = {k: p.detach().cpu() for k, p in model.state_dict().items()}
    stream = io.BytesIO()
    torch.save(weights, stream)
    payload = stream.getvalue()
    digest = hashlib.sha256(payload).hexdigest()
    with tempfile.NamedTemporaryFile(dir=directory, delete=False) as pending:
        temp = Path(pending.name)
    try:
        with temp.open("wb") as f:
            f.write(payload)
            f.flush()
            os.fsync(f.fileno())
        if hashlib.sha256(temp.read_bytes()).hexdigest() != digest:
            raise IOError("Model weight checkpoint verification failed.")
        os.replace(temp, final)
    finally:
        temp.unlink(missing_ok=True)
    return {"filename": final.name, "bytes": len(payload), "sha256": digest}


def train_one_stress(variant, seed, splits, *, steps=5000, device="cuda", weights_dir=None,
                     save_run=None, check_budget=None, progress=None):
    """Use the formal training kernel; evaluate every supplied split except 'train'."""
    held_out = [key for key in splits if key != "train"]
    if not held_out or "train" not in splits:
        raise ValueError("Supply the training split and at least one distinct evaluation split.")
    model, record, device, precision, sync = fit_formal_arm(
        variant, seed, splits, device=device, total_steps=steps,
        check_budget=check_budget, progress=progress
    )
    if weights_dir is not None:
        record["weights"] = _save_stress_weights(model, weights_dir, variant, seed, steps)
    for name in held_out:
        sync()
        before = time.perf_counter()
        record["splits"][name] = capture_blank_predictions(
            model, splits[name], device, precision, batch_size=64, check_budget=check_budget
        )
        sync()
        record["eval_split_seconds"][name] = time.perf_counter() - before
    record["eval_seconds"] = sum(record["eval_split_seconds"].values())
    if save_run:
        save_run(record)
    return record


In [6]:
import base64
import hashlib
import io
import json
import random

import numpy as np


_STRESS_SEED = 20261127
_STRESS_UNITS = ([tuple(9*r+c for c in range(9)) for r in range(9)] +
                 [tuple(9*r+c for r in range(9)) for c in range(9)])
_STRESS_UNITS += [
    tuple(9 * (3 * br + i) + 3 * bc + j for i in range(3) for j in range(3))
    for br in range(3) for bc in range(3)
]
_STRESS_CELL_UNITS = [
    (i//9, 9+i%9, 18+i//27*3+i%9//3) for i in range(81)
]


def _stress_signature(solution):
    """Same full 64-bit symmetry-invariant digest used by the formal split rule."""
    grid = [solution[i:i+9] for i in range(0, 81, 9)]

    def row_pairs(lines):
        fingerprints = []
        for i, left in enumerate(lines):
            for right in lines[i+1:]:
                permutation = dict(zip(left, right))
                unseen = set(range(1, 10))
                lengths = []
                while unseen:
                    x = next(iter(unseen))
                    length = 0
                    while x in unseen:
                        unseen.remove(x)
                        x = permutation[x]
                        length += 1
                    lengths.append(length)
                fingerprints.append(tuple(sorted(lengths)))
        return tuple(sorted(fingerprints))

    signature = sorted((row_pairs(grid), row_pairs(list(zip(*grid)))))
    return int.from_bytes(hashlib.blake2b(repr(signature).encode("ascii"), digest_size=8).digest(),
                          "little")


def _stress_singles(puzzle):
    """Apply naked and row/column/box hidden singles simultaneously until stalled."""
    board = list(map(int, puzzle))
    rounds = 0
    while 0 in board:
        occupied = [0] * 27
        for i, d in enumerate(board):
            if d:
                bit = 1 << (d - 1)
                for u in _STRESS_CELL_UNITS[i]:
                    occupied[u] |= bit
        candidates = [0] * 81
        seen, repeated = [0] * 27, [0] * 27
        for i, d in enumerate(board):
            if d:
                continue
            r, c, b = _STRESS_CELL_UNITS[i]
            mask = 511 & ~(occupied[r] | occupied[c] | occupied[b])
            if not mask:
                raise ValueError("Contradictory Sudoku clues")
            candidates[i] = mask
            for u in (r, c, b):
                repeated[u] |= seen[u] & mask
                seen[u] |= mask
        changed = False
        for i, mask in enumerate(candidates):
            if not mask:
                continue
            forced = mask if mask.bit_count() == 1 else 0
            for u in _STRESS_CELL_UNITS[i]:
                forced |= mask & seen[u] & ~repeated[u]
            if forced:
                if forced.bit_count() != 1:
                    raise ValueError("Inconsistent hidden singles")
                board[i] = forced.bit_length()
                changed = True
        if not changed:
            break
        rounds += 1
    return rounds, board.count(0)


def _stress_cover_count(puzzle):
    """Independent exact-cover verifier. Count solutions, stopping at two."""
    rows = []
    columns = [set() for _ in range(324)]
    for i, given in enumerate(puzzle):
        r, c = divmod(i, 9)
        box = 3*(r//3) + c//3
        for d in range(9):
            if given and given != d+1:
                continue
            covered = (i, 81+9*r+d, 162+9*c+d, 243+9*box+d)
            for j in covered:
                columns[j].add(len(rows))
            rows.append(covered)
    active = dict(enumerate(columns))

    def search():
        if not active:
            return 1
        col = min(active.values(), key=len)
        found = 0
        for option in tuple(col):
            removed = []
            for k in rows[option]:
                previous = active.pop(k)
                removed.append((k, previous))
                for other in previous:
                    for neighbor in rows[other]:
                        if neighbor in active:
                            active[neighbor].remove(other)
            found += search()
            for k, previous in reversed(removed):
                for other in previous:
                    for neighbor in rows[other]:
                        if neighbor in active:
                            active[neighbor].add(other)
                active[k] = previous
            if found >= 2:
                return 2
        return found

    return search()


def _stress_hash(x, y):
    return hashlib.sha256(x.tobytes() + y.tobytes()).hexdigest()


def _stress_profile(x, y, must_stall):
    histogram, stalled_histogram, rounds = {}, {}, []
    clues = np.count_nonzero(x, axis=1)
    for grid, solution, n in zip(x.tolist(), y.tolist(), clues.tolist()):
        if any(sorted(solution[i] for i in unit) != list(range(1,10)) for unit in _STRESS_UNITS):
            raise ValueError("Invalid solution grid")
        if any(a and a != b for a, b in zip(grid, solution)) or _stress_cover_count(grid) != 1:
            raise ValueError("Puzzle verification failed")
        r, stalled = _stress_singles(grid)
        if must_stall and stalled == 0:
            raise ValueError("Singles solved a hard puzzle")
        rounds.append(r)
        histogram[n] = histogram.get(n, 0) + 1
        stalled_histogram[stalled] = stalled_histogram.get(stalled, 0) + 1
    return dict(n_boards=len(x), data_sha256=_stress_hash(x,y),
                clues_min=int(clues.min()), clues_median=float(np.median(clues)),
                clues_mean=float(clues.mean()), clues_max=int(clues.max()),
                clues_histogram=histogram, singles_stalled_histogram=stalled_histogram,
                singles_solved=stalled_histogram.get(0,0),
                singles_rounds_median=float(np.median(rounds)), singles_rounds_max=max(rounds))


def generate_stress_corpus(n=512, seed=_STRESS_SEED):
    """Rebuild the two held-out suites from data_cells.py routines without model access."""
    previous = make_sudoku_splits(n_train=2048, n_val=256, n_test=256, seed=2026)
    prior_x, prior_y = previous["test"]
    expected = "aefcf3dae5d714772666dc2636744a19fe926d42600b4ce9a0de9f7eb0d8505c"
    if _stress_hash(prior_x, prior_y) != expected:
        raise ValueError("Training-era test set does not match its published fingerprint")
    excluded = {_stress_signature(row) for row in prior_y.tolist()}
    if any(x % 10 != 9 for x in excluded):
        raise AssertionError("Fingerprint disagrees with the formal split rule")
    corpus = {}
    report = {"seed": seed, "prior_test_hash": expected, "bucket": 9,
              "excluded_formal_test_signatures":len(excluded), "profiles": {}}
    for offset, suite in enumerate(("original", "hard")):
        rng = random.Random(seed+offset)
        boards, solutions = [], []
        attempts, accepted_solutions = 0, 0
        kept_bands = [0] * 3
        while len(boards) < n:
            solution = _sdk_completion([0]*81, rng=rng)
            puzzle_rng = random.Random(rng.getrandbits(64))
            sig = _stress_signature(solution)
            if sig % 10 != 9 or sig in excluded:
                continue
            accepted_solutions += 1
            if suite == "original":
                band = len(boards) % 3
                board = _sdk_puzzle(solution, puzzle_rng, band)
                attempts += 1
            else:
                # Keep a maximum of one puzzle per previously unseen solution signature.
                # Sample the same three clue bands with equal probability per mask.
                for _ in range(64):
                    band = puzzle_rng.randrange(3)
                    board = _sdk_puzzle(solution, puzzle_rng, band)
                    attempts += 1
                    if _stress_singles(board)[1]:
                        break
                else:
                    continue
            excluded.add(sig)
            kept_bands[band] += 1
            boards.append(board)
            solutions.append(solution)
        x, y = np.asarray(boards,np.uint8), np.asarray(solutions,np.uint8)
        corpus[suite] = (x, y)
        profile = _stress_profile(x, y, suite=="hard")
        profile["eligible_masks_attempted"] = attempts
        profile["eligible_solutions_attempted"] = accepted_solutions
        profile["accepted_initial_clue_bands"] = kept_bands
        report["profiles"][suite] = profile
    return corpus, report


def pack_stress_corpus(corpus, report):
    packed = io.BytesIO()
    arrays = {f"{suite}_{label}": arr for suite, pair in corpus.items()
              for label, arr in zip(("x","y"), pair)}
    encoded_report = json.dumps(report, sort_keys=True, separators=(",",":")).encode("utf-8")
    np.savez_compressed(packed, **arrays, report=np.frombuffer(encoded_report, dtype=np.uint8))
    return base64.b64encode(packed.getvalue()).decode("ascii")


def _read_stress_archive(packed):
    if packed is None:
        packed = globals().get("STRESS_DATA_B64")
    if not packed:
        raise ValueError("Missing embedded Sudoku corpus")
    return np.load(io.BytesIO(base64.b64decode(packed, validate=True)), allow_pickle=False)


def make_stress_sets(packed=None):
    with _read_stress_archive(packed) as archive:
        report = json.loads(archive["report"].tobytes())
        pairs = {}
        for suite in ("original", "hard"):
            x, y = archive[suite+"_x"], archive[suite+"_y"]
            if _stress_hash(x,y) != report["profiles"][suite]["data_sha256"]:
                raise ValueError(f"Corpus digest mismatch for {suite}")
            pairs[suite] = (x, y)
        return pairs


def stress_data_report(packed=None):
    with _read_stress_archive(packed) as archive:
        return json.loads(archive["report"].tobytes())


In [7]:
import numpy as np


_STRESS_MODELS = ("plain", "full_gate", "split_gate")
_STRESS_NAMES = {"plain": "普通循环", "full_gate": "全通道门控", "split_gate": "分区门控"}
_STRESS_ENDPOINTS = ("ever_wrong", "correct32", "ever_correct")


def _stress_rate(n, d):
    return float(n / d) if d else None


def _stress_average(values):
    return None if any(x is None for x in values) else float(np.mean(values))


def _stress_solo(flags, per_board, sizes):
    r8, r32, ever_wrong, ever_correct = flags
    wrong8 = ~r8
    r8_counts, r32_counts = per_board(r8), per_board(r32)
    counts = dict(blanks=int(sizes.sum()), boards=len(sizes), right8=int(r8.sum()),
                  wrong8=int(wrong8.sum()), right32=int(r32.sum()),
                  solved8=int((sizes == r8_counts).sum()), solved32=int((sizes == r32_counts).sum()),
                  lost=int((r8 & ever_wrong).sum()), lost_then_right32=int((r8 & ever_wrong & r32).sum()),
                  fixed32=int((wrong8 & r32).sum()), fixed_ever=int((wrong8 & ever_correct).sum()))
    rates = {key: _stress_rate(counts[n], counts[d]) for key, n, d in (
        ("blank_acc8", "right8", "blanks"), ("blank_acc32", "right32", "blanks"),
        ("board_acc8", "solved8", "boards"), ("board_acc32", "solved32", "boards"),
        ("ever_wrong", "lost", "right8"), ("correct32", "fixed32", "wrong8"),
        ("ever_correct", "fixed_ever", "wrong8"))}
    return {"counts": counts, "rates": rates}


def _stress_matched(split_flags, control_flags, per_board):
    s8, s32, s_lost, s_reached = split_flags
    c8, c32, c_lost, c_reached = control_flags
    right, wrong = s8 & c8, ~s8 & ~c8
    # Dimensions are puzzle × endpoint × [common denominator, split events, control events].
    return np.stack([
        np.stack((per_board(m), per_board(m & s), per_board(m & c)), axis=1)
        for m, s, c in ((right, s_lost, c_lost), (wrong, s32, c32), (wrong, s_reached, c_reached))
    ], axis=1)


def _stress_intervals(counts, resamples, seed):
    # All arms, endpoints and fitted seeds share the sampled puzzle IDs. After
    # resampling, recompute the ratio of cell counts within each seed, then
    # average the three seed-level differences with equal weights.
    n_seeds, n_boards = counts.shape[:2]
    per_board = counts.transpose(1, 0, 2, 3).reshape(n_boards, -1)
    rng = np.random.default_rng(seed)
    distribution = np.full((resamples, len(_STRESS_ENDPOINTS)), np.nan)
    for start in range(0, resamples, 128):
        size = min(128, resamples - start)
        weights = rng.multinomial(n_boards, np.full(n_boards, 1/n_boards), size=size)
        drawn = (weights @ per_board).reshape(size, n_seeds, len(_STRESS_ENDPOINTS), 3)
        den, split, control = (drawn[..., j] for j in range(3))
        diff = np.full(den.shape, np.nan)
        np.divide(split - control, den, where=den != 0, out=diff)
        distribution[start:start + size] = diff.mean(axis=1)
    intervals = []
    for column in distribution.T:
        finite = column[np.isfinite(column)]
        # An undefined replicate signals thin baseline support. Do not drop
        # those draws and silently present a conditional percentile interval.
        interval = (np.quantile(finite, [0.025, 0.975]).tolist()
                    if len(finite) and len(finite) == resamples else None)
        intervals.append((interval, len(finite)))
    return intervals


def _stress_contrast(per_seed_board, control, seeds, resamples, bootstrap_seed):
    counts = np.stack(per_seed_board)
    totals = counts.sum(axis=1)
    intervals = _stress_intervals(counts, resamples, bootstrap_seed)
    metrics = {}
    for j, endpoint in enumerate(_STRESS_ENDPOINTS):
        rows = []
        for seed, (den, split, other) in zip(seeds, totals[:, j, :].tolist()):
            rows.append(dict(seed=seed, shared_blanks=den, split_events=split, control_events=other,
                             split_rate=_stress_rate(split, den), control_rate=_stress_rate(other, den),
                             gap=_stress_rate(split-other, den)))
        mean = {k: _stress_average([r[k] for r in rows]) for k in ("split_rate", "control_rate", "gap")}
        total = {k: sum(r[k] for r in rows) for k in ("shared_blanks", "split_events", "control_events")}
        nets = counts[:, :, j, 1] - counts[:, :, j, 2]
        interval, n_valid = intervals[j]
        metrics[endpoint] = dict(by_seed=rows, mean_over_seeds=mean, counts_across_seeds=total,
                                 boards_with_nonzero_net_gap=int((nets != 0).any(axis=0).sum()),
                                 paired_puzzle_bootstrap_95=interval, valid_bootstrap_draws=n_valid)
    loss_gap = metrics["ever_wrong"]["mean_over_seeds"]["gap"]
    repair_gap = metrics["correct32"]["mean_over_seeds"]["gap"]
    directional = None if loss_gap is None or repair_gap is None else bool(loss_gap < 0 and repair_gap >= 0)
    return dict(left="split_gate", right=control, metrics=metrics,
                observed_lower_loss_without_lower_terminal_correction=directional)


def analyze_stress(records, splits, cohort="hard", *, expected_seeds=(17, 29, 43),
                   expected_steps=5000, bootstrap_samples=4000, bootstrap_seed=2026, decoder=None):
    """Summarize matched cell trajectories; all contrast gaps are split minus control."""
    if decoder is None:
        decoder = globals().get("decode_blank_predictions")
        if decoder is None:
            from stress_train import decode_blank_predictions as decoder
    seeds = tuple(int(s) for s in expected_seeds)
    if not seeds or len(seeds) != len(set(seeds)):
        raise ValueError("Expected seeds must be distinct.")
    if not isinstance(bootstrap_samples, (int, np.integer)) or bootstrap_samples < 0:
        raise ValueError("bootstrap_samples must be a nonnegative integer.")
    if cohort not in ("hard", "original"):
        raise ValueError("Cohort must be hard or original.")
    X, Y = (np.asarray(a) for a in splits[cohort])
    if X.shape != Y.shape or X.ndim != 2 or not len(X) or X.shape[1] != 81:
        raise ValueError("Expected matching nonempty [N,81] Sudoku arrays.")
    blank = X == 0
    sizes = blank.sum(axis=1)
    if np.any(sizes == 0) or np.any((Y < 1) | (Y > 9)):
        raise ValueError("Each board needs at least one blank and a full digit solution.")
    truth = Y[blank]
    offsets = np.cumsum(sizes) - sizes

    def per_board(mask):
        return np.add.reduceat(mask, offsets, dtype=np.int64)

    indexed = {}
    for rec in records:
        key = rec["variant"], int(rec["seed"])
        if key in indexed or rec["steps"] != expected_steps:
            raise ValueError(f"Duplicate or incomplete run {key}.")
        indexed[key] = rec
    if set(indexed) != {(m, s) for m in _STRESS_MODELS for s in seeds}:
        raise ValueError("A complete paired model-by-seed design is required.")

    solorows = {name: [] for name in _STRESS_MODELS}
    paired = {name: [] for name in ("full_gate", "plain")}
    for seed in seeds:
        flags = {}
        for arm in _STRESS_MODELS:
            pred = np.asarray(decoder(indexed[arm, seed]["splits"][cohort], (X, Y)))
            if pred.shape != (32, len(truth)) or pred.dtype.kind not in "iu" or (pred < 1).any() or (pred > 9).any():
                raise ValueError("Decoder must return digit predictions [32, total_blanks].")
            later = pred[8:] == truth
            flags[arm] = (pred[7] == truth, later[-1], ~later.all(axis=0), later.any(axis=0))
            solorows[arm].append(dict(seed=seed, **_stress_solo(flags[arm], per_board, sizes)))
        for control in paired:
            paired[control].append(_stress_matched(flags["split_gate"], flags[control], per_board))
    by_model = {}
    for arm, rows in solorows.items():
        means = {k: _stress_average([r["rates"][k] for r in rows]) for k in rows[0]["rates"]}
        by_model[arm] = {"by_seed": rows, "mean_over_seeds": means}
    contrasts = {name: _stress_contrast(values, name, seeds, bootstrap_samples, bootstrap_seed)
                 for name, values in paired.items()}
    return dict(cohort=cohort, n_boards=len(X), n_blank=int(sizes.sum()), seeds=list(seeds),
                expected_steps=expected_steps, reference_round=8, subsequent_rounds=[9, 32],
                primary_comparison="hard: split_gate minus full_gate", secondary_control="plain",
                equivalence_margin=None, bootstrap_samples=bootstrap_samples, bootstrap_seed=bootstrap_seed,
                inference_scope="Marginal paired puzzle intervals conditional on trained seeds and this cohort.",
                by_model=by_model, contrasts=contrasts)


def print_stress_summary(summary):
    def pct(x):
        return "—" if x is None else f"{100*x:.2f}%"

    print(f"{summary['cohort']}，{summary['n_boards']} 题，{summary['n_blank']} 个原始空格。")
    print("模型           R8整盘   R32整盘    R8空格   R32空格")
    for name, values in summary["by_model"].items():
        m = values["mean_over_seeds"]
        print(f"{_STRESS_NAMES[name]:<8}  " + "  ".join(pct(m[k]) for k in
              ("board_acc8", "board_acc32", "blank_acc8", "blank_acc32")))
    for control, values in summary["contrasts"].items():
        print(f"分区门控减{_STRESS_NAMES[control]}；仅比较第 8 轮两者同对或同错的空格。")
        for name, label in zip(_STRESS_ENDPOINTS, ("同对后曾错", "同错后R32正确", "同错后曾正确")):
            item = values["metrics"][name]
            m = item["mean_over_seeds"]
            counts = item["counts_across_seeds"]
            ci = item["paired_puzzle_bootstrap_95"]
            ci_text = "未计算" if ci is None else f"[{100*ci[0]:+.2f}, {100*ci[1]:+.2f}]"
            gap = "—" if m["gap"] is None else f"{100*m['gap']:+.2f}"
            print(f"  {label}  {pct(m['split_rate'])} / {pct(m['control_rate'])}，差 {gap} 个百分点，"
                  f"95% 区间 {ci_text}。事件 {counts['split_events']} / {counts['control_events']}，"
                  f"共同分母 {counts['shared_blanks']} 个种子–空格。")
    print("各训练种子等权。区间按题配对重采样，只反映这些已训练模型在该题集上的抽样波动。")


In [8]:
import math
import sys
import threading
import time


class BudgetStop(RuntimeError):
    """The experiment must stop and preserve its current results."""


def release_colab_runtime():
    """Release the hosted VM using Colab's public runtime-management function."""
    from google.colab import runtime

    try:
        runtime.unassign()
    except Exception:
        # Retry once in case the backend briefly lost the connection.
        time.sleep(2)
        runtime.unassign()


class RuntimeBudget:
    """Wall-clock guard using the rate shown by the active Colab runtime."""

    def __init__(self, cu_per_hour, already_used_cu=0.0, *, session_start,
                 limit_cu=8.0, max_hours=2.0, reserve_minutes=3.0):
        self.rate = float(cu_per_hour)
        self.prior_cu = float(already_used_cu)
        self.start = float(session_start)
        self.limit_cu = float(limit_cu)
        max_hours = float(max_hours)
        reserve_minutes = float(reserve_minutes)
        values = (self.rate, self.prior_cu, self.start, self.limit_cu,
                  max_hours, reserve_minutes)
        if not all(math.isfinite(value) for value in values):
            raise ValueError("CU 费率、预算和时间都必须是有限数值。")
        if self.rate <= 0 or self.prior_cu < 0:
            raise ValueError("请填写资源面板当前的正数 CU/小时和非负的已用 CU。")
        if not (0 < self.limit_cu <= 8 and 0 < max_hours <= 2 and reserve_minutes > 0):
            raise ValueError("本实验的运行预算上限为 8 CU、2 小时；收尾时间须为正数。")
        if not 0 <= self.start <= time.monotonic():
            raise ValueError("session_start 应取自当前内核首次运行时的 time.monotonic()。")
        available_cu = max(0.0, self.limit_cu - self.prior_cu)
        allowed_seconds = 3600 * min(max_hours, available_cu / self.rate)
        self.release_deadline = self.start + allowed_seconds
        self.work_deadline = self.release_deadline - 60 * reserve_minutes
        self._watchdog = None

    def remaining_seconds(self):
        """Seconds available for training or evaluation before wrap-up begins."""
        return max(0.0, self.work_deadline - time.monotonic())

    def check(self):
        if self.remaining_seconds() == 0:
            raise BudgetStop("已到预留的收尾时间，停止训练并保存当前结果。")

    def summary(self):
        elapsed = max(0.0, time.monotonic() - self.start)
        return {
            "runtime_cu_per_hour_entered": self.rate,
            "previous_sessions_cu_entered": self.prior_cu,
            "this_session_seconds": round(elapsed, 2),
            "estimated_total_cu": round(self.prior_cu + self.rate * elapsed / 3600, 4),
            "software_cu_limit": self.limit_cu,
            "remaining_work_seconds": round(self.remaining_seconds(), 2),
        }

    def arm_watchdog(self):
        """Request VM release at the ceiling even if a training call blocks."""
        if self._watchdog is not None:
            return

        def on_deadline():
            print("已到运行时限，正在释放 Colab 运行时。", file=sys.stderr, flush=True)
            try:
                release_colab_runtime()
            except Exception as exc:
                print(f"Colab 自动释放失败，请手动断开并删除运行时。{exc}",
                      file=sys.stderr, flush=True)

        seconds = max(0.0, self.release_deadline - time.monotonic())
        self._watchdog = threading.Timer(seconds, on_deadline)
        self._watchdog.daemon = True
        self._watchdog.start()

    def disarm_watchdog(self):
        if self._watchdog is not None:
            self._watchdog.cancel()
            self._watchdog = None


In [9]:
from pathlib import Path
import base64
import hashlib
import json
import re
import zlib


def publish_bytes(name, blob):
    """Keep a lossless copy in notebook outputs, independent of Colab VM storage."""
    from IPython.display import HTML, display
    if not re.fullmatch(r'[a-zA-Z0-9_.-]+', name):
        raise ValueError('Invalid archive member name')
    digest = hashlib.sha256(blob).hexdigest()
    packed = zlib.compress(blob, 6)
    # Several small attachments avoid notebook front-end limits for single outputs.
    parts = [packed[i:i+24576] for i in range(0,len(packed),24576)]
    for i, part in enumerate(parts, 1):
        token = f'SUDOKUSTRESS1.{name}.{i}.{len(parts)}.{digest}'
        href = 'data:application/octet-stream;base64,' + base64.b64encode(part).decode('ascii')
        display(HTML(f'<a download="{token}" href="{href}">{name} · {i}/{len(parts)}</a>'))
    return digest


def publish_json(name, value):
    raw = json.dumps(value, ensure_ascii=False, separators=(',', ':'), allow_nan=False).encode('utf-8')
    return publish_bytes(name, raw)


def recover_stress_files(notebook):
    """Read integrity-checked experiment records and weights from an executed .ipynb."""
    if isinstance(notebook, (str, Path)):
        notebook = json.loads(Path(notebook).read_text(encoding='utf-8-sig'))
    fragments = {}
    regex = re.compile(r'download="SUDOKUSTRESS1\.([a-zA-Z0-9_.-]+)\.(\d+)\.(\d+)\.([0-9a-f]{64})" href="data:application/octet-stream;base64,([a-zA-Z0-9+/=]+)"')
    for cell in notebook['cells']:
        for output in cell.get('outputs', []):
            html = ''.join(output.get('data', {}).get('text/html', ''))
            for name, index, total, digest, payload in regex.findall(html):
                index, total = int(index), int(total)
                if not 1 <= index <= total:
                    raise ValueError(f'Invalid part index for {name}')
                parts = fragments.setdefault((name,total,digest), {})
                chunk = base64.b64decode(payload, validate=True)
                if index in parts and parts[index] != chunk:
                    raise ValueError(f'Conflicting part {index} for {name}')
                parts[index] = chunk
    recovered = {}
    for (name,total,digest), parts in fragments.items():
        if len(parts) != total:
            continue
        raw = zlib.decompress(b''.join(parts[i] for i in range(1,total+1)))
        if hashlib.sha256(raw).hexdigest() != digest or (name in recovered and recovered[name] != raw):
            raise ValueError(f'Invalid or ambiguous archive member {name}')
        recovered[name] = raw
    return recovered


In [10]:
import hashlib
import json
import time
import threading
import traceback
from pathlib import Path


def stress_setup(rate_override, previous_cu):
    watchdog = None
    try:
        if not torch.cuda.is_available():
            raise RuntimeError('本实验需要带 CUDA 的 Colab GPU 运行时。')
        rate = float(rate_override)
        if rate == 0:
            watchdog = threading.Timer(180, release_colab_runtime)
            watchdog.daemon = True
            watchdog.start()
            rate = float(input('输入资源面板上当前会话的 CU/小时，再按回车：').strip())
        budget = RuntimeBudget(rate, previous_cu, session_start=STRESS_SESSION_START)
        budget.arm_watchdog()
        if watchdog:
            watchdog.cancel()
        budget.check()
        train_split = make_sudoku_splits(n_train=2048,n_val=256,n_test=256,seed=2026)['train']
        hash_train = hashlib.sha256(train_split[0].tobytes() + train_split[1].tobytes()).hexdigest()
        if hash_train != PRIOR_TRAIN_SHA:
            raise ValueError('训练数据哈希与第一轮正式实验不符。')
        cohorts = make_stress_sets()
        split_hashes = {k:hashlib.sha256(x.tobytes()+y.tobytes()).hexdigest() for k,(x,y) in cohorts.items()}
        assert set(cohorts) == {'original','hard'}
        assert all(len(x)==512 and x.shape==y.shape==(512,81) for x,y in cohorts.values())
        protocol = dict(seeds=[17,29,43], models=['plain','full_gate','split_gate'], train_steps=5000,
                        train_loops=8, eval_loops=list(range(1,33)), batch=64, source_sha256=SOURCE_SHA,
                        training_sha256=hash_train, evaluation_sha256=split_hashes, data=stress_data_report(),
                        gpu=torch.cuda.get_device_name(), torch_version=str(torch.__version__),
                        primary='Hard cohort, split_gate minus full_gate, matched at R8; any loss by R32 and terminal repair at R32.',
                        secondary='Compare against plain; repeat on original cohort; report any repair and transient changes.')
        where = Path('/content/sudoku_stress_weights')
        where.mkdir(exist_ok=True)
        publish_json('protocol.json',protocol)
        print('STRESS_READY',protocol['gpu'],rate,'CU/小时；原分布和难题各 512 道。',flush=True)
        return dict(splits={'train':train_split,**cohorts}, budget=budget, runs=[], metadata=protocol, weights=where)
    except BaseException:
        traceback.print_exc()
        release_colab_runtime()
        raise
    finally:
        if watchdog:
            watchdog.cancel()


def stress_run(ctx, variant, seed):
    try:
        budget = ctx['budget']
        budget.check()
        for existing in ctx['runs']:
            if (existing['variant'],existing['seed']) == (variant,seed):
                print(f'内核中已有完成记录：{variant}，种子 {seed}。', flush=True)
                return
        def progress(model, run_seed, step, total):
            print(f'STRESS_STEP {model} {run_seed} {step}/{total}', flush=True)
        print(f'STRESS_START {variant} {seed}', flush=True)
        rec = train_one_stress(variant,seed,ctx['splits'],steps=5000,weights_dir=ctx['weights'],
                               check_budget=budget.check,progress=progress)
        assert rec['steps']==5000 and set(rec['splits'])=={'original','hard'}
        weight_name = f'sudoku-{variant}-seed{seed}-steps5000.pt'
        blob = (ctx['weights']/weight_name).read_bytes()
        publish_json(f'run-{variant}-{seed}.json',rec)
        publish_bytes(weight_name,blob)
        ctx['runs'].append(rec)
        print(f'STRESS_SAVED {len(ctx["runs"])}/9 · {variant} · {seed} · 5000 步 · 32 轮逐格预测已保存', flush=True)
    except BaseException:
        print('STRESS_RUN_FAILED',variant,seed, flush=True)
        traceback.print_exc()
        time.sleep(2)
        release_colab_runtime()
        raise


def stress_finish(ctx):
    try:
        assert len(ctx['runs'])==9
        summaries = {}
        for cohort in ('hard','original'):
            result = analyze_stress(ctx['runs'],ctx['splits'],cohort=cohort)
            print_stress_summary(result)
            summaries[cohort] = result
        summary = dict(status='complete',budget=ctx['budget'].summary(),protocol=ctx['metadata'],cohorts=summaries)
        publish_json('stress_summary.json',summary)
        print('STRESS_COMPLETE 9/9 · 两组逐格轨迹和九组权重已保存。',flush=True)
    except BaseException:
        traceback.print_exc()
        raise
    finally:
        time.sleep(8)
        release_colab_runtime()
        ctx['budget'].disarm_watchdog()


In [11]:
# @title 已固定的两组题面、答案和独立求解器报告
STRESS_DATA_B64 = (
    '...'
    'IAEAAIITAQAAAA=='
)


In [12]:
# @title 初始化数据和计时
CU_PER_HOUR = 0.0  # @param {type:"number"}
PREVIOUS_CU = 0.0  # @param {type:"number"}
SOURCE_SHA = 'acbb251acdb41215f0b9712b3ad5481120035dac1dfbb9551a4fae2c21e14598'
PRIOR_TRAIN_SHA = '1c0486bb62b1bfb16d112ffc33b44c2f4ad2775149a84d49d28f41f768decd6b'
stress_context = stress_setup(CU_PER_HOUR, PREVIOUS_CU)


输入资源面板上当前会话的 CU/小时，再按回车：2.33


STRESS_READY NVIDIA L4 2.33 CU/小时；原分布和难题各 512 道。


In [13]:
# @title plain · 种子 17
stress_run(stress_context, 'plain', 17)


STRESS_START plain 17
STRESS_STEP plain 17 1000/5000
STRESS_STEP plain 17 2000/5000
STRESS_STEP plain 17 3000/5000
STRESS_STEP plain 17 4000/5000
STRESS_STEP plain 17 5000/5000


STRESS_SAVED 1/9 · plain · 17 · 5000 步 · 32 轮逐格预测已保存


In [14]:
# @title full_gate · 种子 17
stress_run(stress_context, 'full_gate', 17)


STRESS_START full_gate 17
STRESS_STEP full_gate 17 1000/5000
STRESS_STEP full_gate 17 2000/5000
STRESS_STEP full_gate 17 3000/5000
STRESS_STEP full_gate 17 4000/5000
STRESS_STEP full_gate 17 5000/5000


STRESS_SAVED 2/9 · full_gate · 17 · 5000 步 · 32 轮逐格预测已保存


In [15]:
# @title split_gate · 种子 17
stress_run(stress_context, 'split_gate', 17)


STRESS_START split_gate 17
STRESS_STEP split_gate 17 1000/5000
STRESS_STEP split_gate 17 2000/5000
STRESS_STEP split_gate 17 3000/5000
STRESS_STEP split_gate 17 4000/5000
STRESS_STEP split_gate 17 5000/5000


STRESS_SAVED 3/9 · split_gate · 17 · 5000 步 · 32 轮逐格预测已保存


In [16]:
# @title full_gate · 种子 29
stress_run(stress_context, 'full_gate', 29)


STRESS_START full_gate 29
STRESS_STEP full_gate 29 1000/5000
STRESS_STEP full_gate 29 2000/5000
STRESS_STEP full_gate 29 3000/5000
STRESS_STEP full_gate 29 4000/5000
STRESS_STEP full_gate 29 5000/5000


STRESS_SAVED 4/9 · full_gate · 29 · 5000 步 · 32 轮逐格预测已保存


In [17]:
# @title split_gate · 种子 29
stress_run(stress_context, 'split_gate', 29)


STRESS_START split_gate 29
STRESS_STEP split_gate 29 1000/5000
STRESS_STEP split_gate 29 2000/5000
STRESS_STEP split_gate 29 3000/5000
STRESS_STEP split_gate 29 4000/5000
STRESS_STEP split_gate 29 5000/5000


STRESS_SAVED 5/9 · split_gate · 29 · 5000 步 · 32 轮逐格预测已保存


In [18]:
# @title plain · 种子 29
stress_run(stress_context, 'plain', 29)


STRESS_START plain 29
STRESS_STEP plain 29 1000/5000
STRESS_STEP plain 29 2000/5000
STRESS_STEP plain 29 3000/5000
STRESS_STEP plain 29 4000/5000
STRESS_STEP plain 29 5000/5000


STRESS_SAVED 6/9 · plain · 29 · 5000 步 · 32 轮逐格预测已保存


In [19]:
# @title split_gate · 种子 43
stress_run(stress_context, 'split_gate', 43)


STRESS_START split_gate 43
STRESS_STEP split_gate 43 1000/5000
STRESS_STEP split_gate 43 2000/5000
STRESS_STEP split_gate 43 3000/5000
STRESS_STEP split_gate 43 4000/5000
STRESS_STEP split_gate 43 5000/5000


STRESS_SAVED 7/9 · split_gate · 43 · 5000 步 · 32 轮逐格预测已保存


In [20]:
# @title plain · 种子 43
stress_run(stress_context, 'plain', 43)


STRESS_START plain 43
STRESS_STEP plain 43 1000/5000
STRESS_STEP plain 43 2000/5000
STRESS_STEP plain 43 3000/5000
STRESS_STEP plain 43 4000/5000
STRESS_STEP plain 43 5000/5000


STRESS_SAVED 8/9 · plain · 43 · 5000 步 · 32 轮逐格预测已保存


In [21]:
# @title full_gate · 种子 43
stress_run(stress_context, 'full_gate', 43)


STRESS_START full_gate 43
STRESS_STEP full_gate 43 1000/5000
STRESS_STEP full_gate 43 2000/5000
STRESS_STEP full_gate 43 3000/5000
STRESS_STEP full_gate 43 4000/5000
STRESS_STEP full_gate 43 5000/5000


STRESS_SAVED 9/9 · full_gate · 43 · 5000 步 · 32 轮逐格预测已保存


In [ ]:
# @title 汇总两套题的配对指标并释放 GPU
stress_finish(stress_context)


hard，512 题，24767 个原始空格。
模型           R8整盘   R32整盘    R8空格   R32空格
普通循环      4.49%  41.86%  77.94%  81.77%
全通道门控     6.05%  45.12%  78.20%  82.69%
分区门控      5.60%  44.60%  78.38%  82.80%
分区门控减全通道门控；仅比较第 8 轮两者同对或同错的空格。
  同对后曾错  4.90% / 5.49%，差 -0.59 个百分点，95% 区间 [-0.98, -0.19]。事件 2567 / 2876，共同分母 52439 个种子–空格。
  同错后R32正确  30.35% / 30.98%，差 -0.63 个百分点，95% 区间 [-2.93, +1.72]。事件 3152 / 3218，共同分母 10394 个种子–空格。
  同错后曾正确  34.97% / 36.19%，差 -1.22 个百分点，95% 区间 [-3.47, +1.04]。事件 3632 / 3759，共同分母 10394 个种子–空格。
分区门控减普通循环；仅比较第 8 轮两者同对或同错的空格。
  同对后曾错  4.97% / 5.31%，差 -0.34 个百分点，95% 区间 [-0.74, +0.07]。事件 2598 / 2775，共同分母 52294 个种子–空格。
  同错后R32正确  30.94% / 27.42%，差 +3.53 个百分点，95% 区间 [+1.23, +5.73]。事件 3229 / 2863，共同分母 10445 个种子–空格。
  同错后曾正确  35.45% / 32.71%，差 +2.74 个百分点，95% 区间 [+0.61, +4.96]。事件 3700 / 3416，共同分母 10445 个种子–空格。
各训练种子等权。区间按题配对重采样，只反映这些已训练模型在该题集上的抽样波动。
original，512 题，21190 个原始空格。
模型           R8整盘   R32整盘    R8空格   R32空格
普通循环      80.73%  94.47%  97.18%  98.70%
全通道门控     80.66%  95.25%  97.29%  

STRESS_COMPLETE 9/9 · 两组逐格轨迹和九组权重已保存。
